# **YOLOv8 Full Training & Hyperparameter Optimization Pipeline**

---

### FOR GOOGLE COLAB ONLY
This notebook is designed as a standardized environment for testing and evaluating the YOLO models in Google Colab.

### **IMPORTANT: SET RUNTIME TO T4 GPU**
Before starting, ensure you are using the correct hardware accelerator:
1. Go to **Runtime** (Środowisko wykonawcze) -> **Change runtime type** (Zmień typ środowiska wykonawczego).
2. Under **Hardware accelerator**, select **T4 GPU**.
3. Click **Save**.

*Training on CPU will be extremely slow and may fail.*

---

## **How to Run This Notebook**

1. **Run 1. SETUP** (1.1 SETUP & 1.2 SETUP)
   * **IMPORTANT!!!** Upload **dataset_fanal.zip** to the "acne-data" folder on your Google Drive
   * **WARNING!** The session may restart or fail here:
      ```python
      import condacolab
      condacolab.install()
      ```
      **This is expected. Just run the cells below.**

   * At one point you will be asked for permission to access your Drive. This is mandatory to save your progress so you don't lose data when the session expires.

2. **Run 2. ALL FUNCTIONS**
   * Run all function definitions.
   * If you want to see how session recovery works, run the Smoke Test (`fast_smoke_test()`)

3. **Configure flags in `main()` in 3. MAIN**
   * Open the `main()` function.
   * Set flags according to your needs:
     * `RUN_VALIDATION`
     * `RUN_PRODUCTION`
     * `RUN_HPARAM_SEARCH`
     * etc.

4. **Run 3. MAIN**
   * Execute the final cell to start the pipeline.
   * If Colab disconnects (and it most certainly will), simply run this cell again to pick up exactly where you left off (after running setup and functions once again).

# **1.1 SETUP (before crash)**

## Clone repo

In [ ]:
import os

REPO_URL = "https://github.com/kenami0981/DermaAI.git"
REPO_NAME = "DermaAI"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
%cd {REPO_NAME}
!ls

## Set up the Conda environment

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install() # Here session will fail but it is OK, just run cells below

# **1.2 SETUP (after crash)**

In [ ]:
YAML_PATH = "Models/environment.yml"

!conda env create -f DermaAI/Models/environment.yml # it will take few minutes

In [ ]:
# Use this command when you modified environment.yml
# It updates the existing conda environment and removes packages no longer listed

# Then remember to run cell below !!!

# !conda env update -n acne-detection -f DermaAI/Models/environment.yml --prune

In [ ]:
import sys
import os

env_path = "/usr/local/envs/acne-detection/lib/python3.10/site-packages"

if os.path.exists(env_path):
    sys.path.append(env_path)
    print("Conda environment successfully linked to the notebook.")
else:
    print(f"ERROR: Environment path not found: {env_path}")

In [ ]:
# !conda run -n acne-detection python main.py # jeśli by tak odpalać trening z pliku .py, nie trzeba wtedy ustawiać ścieżek

## Download the dataset

In [ ]:
# IMPORTANT!!! Upload your dataset .zip to the "acne-data" folder on your Google Drive.
# (If you use a different folder, remember to update the path)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Assuming the file is named "dataset_final.zip"
# (if not, make sure to change the name in the path below):

!cp /content/drive/MyDrive/acne-data/dataset_final_preprocessed.zip /content/

In [ ]:
!unzip -q /content/dataset_final_preprocessed.zip -d /content/DermaAI/Models/data

In [ ]:
# Uncomment if you want to unmount your Drive after you are done for the day (but it will be disconnected anyway after some time)

# drive.flush_and_unmount()

# **2. ALL FUNCTIONS**

## All imports

In [ ]:
import os
import gc
import yaml
import torch
import optuna
import pandas as pd
import zoneinfo
import shutil

from pathlib import Path
from datetime import datetime
from ultralytics import YOLO


## Paths Configuration

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/DermaAI/Models").resolve()
LOCAL_ROOT = Path("/content/DermaAI/Models").resolve()
DATA_DIR = LOCAL_ROOT / "data" / "dataset_final_preprocessed"
DATA_YAML = DATA_DIR / "data.yaml"

MODELS_DIR = DRIVE_ROOT / "models"
YOLO_WEIGHTS = MODELS_DIR / "yolov8s.pt"
RUNS_DIR = DRIVE_ROOT / "runs"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

for folder in [DATA_DIR, MODELS_DIR, RUNS_DIR, DRIVE_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def validate_project_paths():
    """
    Validates project structure and normalizes dataset config.

    Validation rules:
    - DATA_YAML is required
    """

    print("\n=== PROJECT VALIDATION ===\n")

    paths = {
        "DRIVE_ROOT": DRIVE_ROOT,
        "LOCAL_ROOT": LOCAL_ROOT,
        "DATA_DIR": DATA_DIR,
        "DATA_YAML": DATA_YAML,
        "MODELS_DIR": MODELS_DIR,
        "RUNS_DIR": RUNS_DIR
    }

    valid = True

    for name, path in paths.items():

        status = "OK" if path.exists() else "MISSING"

        print(f"{name:<15}: {path}")
        print(f"{'':<15}  Status: {status}\n")

        # Only dataset config is critical
        if name == "DATA_YAML" and not path.exists():
            valid = False

    # Normalize dataset paths automatically
    if DATA_YAML.exists():

        try:
            with open(DATA_YAML, "r") as f:
                config = yaml.safe_load(f)

            config["train"] = str(DATA_DIR / "train" / "images")
            config["val"] = str(DATA_DIR / "valid" / "images")
            config["test"] = str(DATA_DIR / "test" / "images")

            with open(DATA_YAML, "w") as f:
                yaml.dump(config, f, default_flow_style=False)

            print("data.yaml paths normalized.\n")

        except Exception as e:
            print(f"Failed to normalize data.yaml: {e}")
            valid = False

    return valid

In [ ]:
def get_timestamp():
    pl_timezone = zoneinfo.ZoneInfo("Europe/Warsaw")
    return datetime.now(pl_timezone).strftime("%Y-%m-%d_%H-%M")

## **SMOKE TEST Instruction**

**Step 0: Preparation**

Uncomment the last two lines of the code below!

**Step 1: The "Crash"**

Run the code. Let it finish 3 or 4 trials, then **click the STOP button** in Colab (or "Przerwij wykonywanie kodu"). This simulates a session timeout or a manual interruption.

**Step 2: The "Recovery"**
Run the code again. Look at the first line of the output.
*   It will say: `>>> PROGRESS RECOVERED: 4 trials found`.
*   **Result:** This proves the system is OK - it didn't start over - it simply picked up where it left off.



In [ ]:
import optuna
import time
from pathlib import Path
from google.colab import drive

def fast_smoke_test():

    # 1. Mount Drive
    drive.mount('/content/drive', force_remount=True)

    # 2. Define the database path on Drive
    db_path = "/content/drive/MyDrive/fast_test.db"

    # 3. Initialize/Load study
    study = optuna.create_study(
        study_name="fast_recovery_test",
        storage=f"sqlite:///{db_path}",
        load_if_exists=True,
        direction="minimize"
    )

    print(f"\n>>> PROGRESS RECOVERED: {len(study.trials)} trials found in database.")

    # 4. Simple math (simulating work)
    def objective(trial):
        time.sleep(2)  # Pause to have time to "crash" it
        return (trial.suggest_float("x", 0, 10) - 5)**2

    print(">>> Starting 20 trials. Interrupt whenever you want!")
    study.optimize(objective, n_trials=20)

# # Uncomment these 2 lines below if you want to perform the recovery test

# if __name__ == "__main__":
#     fast_smoke_test()

## TRAINING

### IMPORTANT! Make sure the T4 GPU runtime is enabled
This script requires a GPU for efficient training. Go to Runtime (Środowisko wykonawcze) --> Change runtime type (Zmień typ środowiska wykonawczego) and select T4 GPU

### EVEN MORE IMPORTANT:

info for Colab Free (T4 GPU)
* **Max Lifetime:** Up to **12 hours** (but it crashed after 4 in my case).
* **Idle Timeout:** **30–90 minutes**. If you stop interacting or close the tab, the session dies quickly.
* **Availability:** Google can reclaim the GPU at any time if demand is high.
* **Storage:** All local files are **deleted** once the session ends. Make sure to save everything on google drive or your device

In [ ]:
TRAINING_RUN_NAME = f"acne_train_production_v1"


def merge_best_params(base_params, best_params):
    merged = base_params.copy()
    merged.update(best_params or {})
    return merged


def train_production(best_params=None):

    # BASE TRAINING CONFIG 
    train_params = {

        "data": str(DATA_YAML),
        "project": str(RUNS_DIR),
        "name": TRAINING_RUN_NAME,
        "exist_ok": True,

        "plots": True,
        "verbose": True,

        "device": 0 if torch.cuda.is_available() else "cpu",
        "workers": 4,
        "batch": -1 if torch.cuda.is_available() else 8,

        "epochs": 150,
        "patience": 20,

        "imgsz": 640,

        "cos_lr": True,

        "save": True,
        "save_period": 1
    }


    # MERGE OPTUNA BEST PARAMS

    if best_params:
      train_params = merge_best_params(train_params, best_params)

    for k, v in train_params.items():
      print(f"{k}: {v}")
    print(f" ")


    # RESUME LOGIC

    run_dir = RUNS_DIR / TRAINING_RUN_NAME
    ckpt_path = run_dir / "weights" / "last.pt"

    resume_training = ckpt_path.exists()

    if resume_training:
        print(f"[RESUME] Found checkpoint: {ckpt_path}")
        model = YOLO(str(ckpt_path))
        train_params["resume"] = True
    else:
        print("[START] No checkpoint found, starting fresh")
        model = YOLO(str(YOLO_WEIGHTS))
        train_params["resume"] = False




    # TRAINING EXECUTION

    try:
        results = model.train(**train_params)

    except KeyboardInterrupt:
        print("[INTERRUPTED] Training stopped by user")
        raise

    except Exception as e:
        print(f"[ERROR] Training failed: {e}")
        raise



    # SAVE TRAINING CONFIG

    weights_dir = run_dir / "weights"
    config_path = weights_dir / "training_params.yaml"

    with open(config_path, "w") as f:
        yaml.dump(train_params, f)

    print("\nTraining complete.")
    print(f"Saved to: {weights_dir}")

    return results

## EVALUATION

In [ ]:
def get_metrics_from_csv(run_name):

    csv_path = RUNS_DIR / run_name / "results.csv"

    if not csv_path.exists():
        print("results.csv not found.")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    best = df.loc[df["metrics/mAP50(B)"].idxmax()]

    print("\n=== BEST METRICS ===\n")
    print(f"Epoch      : {int(best['epoch'])}")
    print(f"mAP50      : {best['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95   : {best['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision  : {best['metrics/precision(B)']:.4f}")
    print(f"Recall     : {best['metrics/recall(B)']:.4f}")

In [ ]:
def show_per_class_metrics(run_name):

    model_path = RUNS_DIR / run_name / "weights" / "best.pt"

    if not model_path.exists():
        print("Model not found.")
        return

    model = YOLO(str(model_path))
    metrics = model.val(data=str(DATA_YAML), verbose=False)

    print("\n=== PER-CLASS METRICS ===\n")

    for i, name in model.names.items():
        print(f"{name}")
        print(f"  mAP50     : {metrics.box.ap50[i]:.4f}")
        print(f"  mAP50-95  : {metrics.box.ap[i]:.4f}")
        print(f"  Precision : {metrics.box.p[i]:.4f}")
        print(f"  Recall    : {metrics.box.r[i]:.4f}")
        print()

### **EVALUATION GUIDE**

| Term | Description | Simple Interpretation |
| :--- | :--- | :--- |
| **Precision** | Accuracy of detections | How many detected items were actually correct? |
| **Recall** | Ability to find objects | How many of the total objects did the model find? |
| **mAP50** | Mean Average Precision | Main metric. Success rate at 50% overlap (IoU). |
| **mAP50-95** | Strict mAP | Measures how perfectly the boxes fit the objects. |

### **YOLO Model Benchmarks**

| Metric | Baseline (Poor) | Target (Good) | Perfect |
| :--- | :--- | :--- | :--- |
| **mAP50** | < 0.30 | 0.50 - 0.70 | > 0.85 |
| **mAP50-95** | < 0.15 | 0.30 - 0.45 | > 0.60 |
| **Precision** | < 0.40 | 0.70 - 0.80 | > 0.90 |
| **Recall** | < 0.30 | 0.60 - 0.75 | > 0.85 |


## Hyperparameter Optimization

In [ ]:
def run_hyperparameter_search():

    # Use a fixed study name (no timestamp) to ensure Colab sessions run OK
    study_name = f"acne_hparam_search_v2"

    def objective(trial):

        model = None
        results = None

        params = {

            "data": str(DATA_YAML),
            "project": str(RUNS_DIR),
            "name": f"{study_name}_trial_{trial.number}",
            "exist_ok": True,

            "device": 0,
            "workers": 4,
            "batch": -1,

            "epochs": 50, # s learns slower than n
            "patience": 10,
            "imgsz": 640,


            "optimizer": "AdamW", # change to AdamW besause it was better in yolov8n

            # Optuna will suggest hyperparameters for each trial
            # Learning speed, 'log=True' helps Optuna jump between small and large scales
            # "lr0": trial.suggest_float("lr0", 1e-5, 5e-3, log=True),
            "lr0": trial.suggest_float("lr0", 1e-5, 2e-3, log=True),

            # Overfitting protection
            # "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),

            # "mosaic": trial.suggest_float("mosaic", 0.0, 0.7),
            # "mixup": trial.suggest_float("mixup", 0.0, 0.05),
            "mosaic": trial.suggest_float("mosaic", 0.0, 0.6),
            "mixup": trial.suggest_float("mixup", 0.0, 0.1),

            # Color
            # "hsv_s": trial.suggest_float("hsv_s", 0.2, 0.7),
            # "hsv_v": trial.suggest_float("hsv_v", 0.1, 0.5),
            "hsv_s": trial.suggest_float("hsv_s", 0.0, 0.4), # smaller because of the CLAHE
            "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.3),

            # Zoom
            # "scale": trial.suggest_float("scale", 0.2, 0.6),
            "scale": trial.suggest_float("scale", 0.3, 0.7),

            "degrees": trial.suggest_float("degrees", 0.0, 30.0),

            "save": False, # Is set False to save space on Google Drive
            "verbose": False,
            "plots": False
        }

        try:
            model = YOLO(str(YOLO_WEIGHTS))
            results = model.train(**params)

            metrics = results.results_dict or {}
            return metrics.get("metrics/mAP50(B)", 0.0) # metrics/mAP50-95(B) was too harsh

        except Exception as e:
            print(f"Trial failed: {e}")
            return 0.0

        finally:
            # Memory cleanup to prevent Out-Of-Memory errors in Colab
            if model is not None:
                del model
            if results is not None:
                del results

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()

    # Define the database path on Google Drive
    db_path = RUNS_DIR / f"{study_name}.db"
    storage_url = f"sqlite:///{db_path}"

    print(f"Data Base Optuna: {db_path}")

    study = optuna.create_study(
        study_name=study_name,
        storage=storage_url,
        load_if_exists=True, # !!! allows the study to resume after a crash or disconnection !!!
        direction="maximize",
        pruner=optuna.pruners.MedianPruner()
    )

    # CHange Running into failed
    for t in study.trials:
        if t.state == optuna.trial.TrialState.RUNNING:
            print(f"Fixing stale RUNNING trial {t.number}")
            study.tell(trial=t.number, state=optuna.trial.TrialState.FAIL)

    print("\n=== TRIAL STATUS ===")

    for t in study.trials:
        print(f"Trial {t.number}: {t.state.name}")

    completed = len([t for t in study.trials if t.state.name == "COMPLETE"])
    failed = len([t for t in study.trials if t.state.name == "FAIL"])
    pruned = len([t for t in study.trials if t.state.name == "PRUNED"])

    print(f"\nCompleted: {completed}")
    print(f"Failed: {failed}")
    print(f"Pruned: {pruned}")

    TARGET_TRIALS = 12
    remaining = max(0, TARGET_TRIALS - completed)

    print(f"Remaining trials to run: {remaining}")

    if remaining > 0:
        study.optimize(objective, n_trials=remaining)
    else:
        print("Search already complete.")

    best_path = RUNS_DIR / f"{study_name}_best.yaml"

    with open(best_path, "w") as f:
        yaml.dump(study.best_params, f)

    print("\nBest parameters:")
    print(study.best_params)

    return study.best_params

## SAVE MODEL AND WEIGHTS

In [ ]:
import shutil
from pathlib import Path


def backup_to_drive(run_name, unmount=True):
    """
    Export final critical artifacts to dedicated archive folder.

    ZMIANA:
    - nie kopiuje całego runa
    - kopiuje tylko kluczowe pliki
    - szybsze
    """

    drive_root = Path("/content/drive/MyDrive")

    if not drive_root.exists():
        print("Google Drive is not mounted.")
        return

    archive_root = drive_root / "acne-archive"
    archive_root.mkdir(parents=True, exist_ok=True)

    run_dir = RUNS_DIR / run_name
    weights_dir = run_dir / "weights"

    if not run_dir.exists():
        print(f"Run not found: {run_dir}")
        return

    export_dir = archive_root / run_name
    export_dir.mkdir(parents=True, exist_ok=True)

    files_to_copy = [
        weights_dir / "best.pt",
        weights_dir / "last.pt",
        run_dir / "results.csv",
        weights_dir / "training_params.yaml"
    ]

    print(f"\nExporting run: {run_name}")

    for file in files_to_copy:
        if file.exists():
            shutil.copy2(file, export_dir / file.name)
            print(f"Copied: {file.name}")

    print("\nExport complete.")

    if unmount:
        try:
            from google.colab import drive
            drive.flush_and_unmount()
            print("Drive successfully unmounted.")

        except Exception as e:
            print(f"Could not unmount Drive: {e}")

# **3. MAIN**

### MAIN EXECUTION PIPELINE

In [ ]:
def main():
    """
    Main project execution controller.

    """

    # !!! Configure flags here: !!!

    # Validate project structure and required files.
    RUN_VALIDATION = True # Best to keep it True permanently
    # Run Optuna hyperparameter optimization
    RUN_HPARAM_SEARCH = True
    # Run full production training
    RUN_PRODUCTION = False
    # Display overall training metrics from results.csv
    SHOW_METRICS = False
    # Display detailed per-class evaluation metrics
    SHOW_PER_CLASS = False
    # Backup selected run to Google Drive
    BACKUP_TO_DRIVE = False
    # Set True if this is the final cell and you want to close the connection to Drive
    UNMOUNT_DRIVE = False



    # Existing run name
    # Required for evaluation / backup
    RUN_NAME = TRAINING_RUN_NAME

    best_params = None
    best_params_path = RUNS_DIR / "acne_hparam_search_v2_best.yaml"

    if best_params_path.exists():
        with open(best_params_path, "r") as f:
            best_params = yaml.safe_load(f)

    # Validation
    if RUN_VALIDATION:
        if not validate_project_paths():
            print("Validation failed.")
            return

    # Hyperparameter search
    if RUN_HPARAM_SEARCH:
        best_params = run_hyperparameter_search()

    # Production training
    if RUN_PRODUCTION:
        results = train_production(best_params)

        if results:
            produced_run_name = results.save_dir.name


    target_run = produced_run_name or RUN_NAME

    # Metrics
    if SHOW_METRICS:
        get_metrics_from_csv(target_run)

    # Per-class validation
    if SHOW_PER_CLASS:
        show_per_class_metrics(target_run)

    # Backup training (no need for Hyperparameter search)
    if BACKUP_TO_DRIVE:
        backup_to_drive(
            target_run,
            unmount=UNMOUNT_DRIVE
        )





# ENTRYPOINT
if __name__ == "__main__":
    main()